<a href="https://colab.research.google.com/github/Sahilgulati2006/ai-sys-des/blob/main/01-model-apis/03-streaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Streaming

**Goal:** Stream responses and understand what UIs need from a streaming backend.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


## Setup

Each notebook is self-contained, so the next two cells stand it up from scratch:

1. **Install dependencies.** The `aien` package (this repo) carries the shared setup helper and pulls in the `groq` client, the only dependency this notebook needs.
2. **Load your API key.** Get a free key at [console.groq.com](https://console.groq.com/) (no credit card). In Colab, add it via the **key icon** in the left sidebar → **Add new secret**, name it exactly `GROQ_API_KEY`, paste the value, and toggle **Notebook access** on. Running locally instead? Set `GROQ_API_KEY` as an environment variable.

(Full walkthrough and model-picking guidance live in [00-setup/00-environment.ipynb](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb).)

In [1]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 10.4 MB/s eta 0:00:00


In [2]:
from aien import setup

# Loads GROQ_API_KEY (Colab Secrets or local env var) and returns a ready
# Groq client. Pass model=... to override the default; if a call later 404s,
# list available models — see 00-setup/00-environment.ipynb.
client, MODEL = setup()

Groq client ready. MODEL = openai/gpt-oss-120b


## Why streaming

Two latencies matter in an LLM product, and they're wildly different:

- **Time to first token (TTFT):** how long until the user sees *anything*. Typically well under a second on Groq (which uses custom inference hardware).
- **Total latency:** how long until the response is complete. For a long answer, multiple seconds.

> **💡 Why it matters —** a non-streaming UI makes the user stare at a spinner for the total; a streaming UI feels responsive after the TTFT. Same model, same cost, entirely different product.


In [3]:
import time

start = time.monotonic()
first_token_at = None

stream = client.chat.completions.create(
    model=MODEL,
    max_tokens=400,
    stream=True,
    messages=[{'role': 'user', 'content': 'Explain, in ~150 words, why database indexes speed up reads but slow down writes.'}],
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        if first_token_at is None:
            first_token_at = time.monotonic() - start
        print(delta, end='', flush=True)

total = time.monotonic() - start
print(f'\n\nTTFT: {first_token_at:.2f}s   total: {total:.2f}s')


Database indexes are auxiliary data structures—typically B‑trees, hash maps, or bitmaps—that store a sorted (or otherwise searchable) copy of one or more columns. When a query needs to locate rows by those columns, the engine can traverse the index instead of scanning the whole table, reducing I/O from *O(N)* to *O(log N)* (or O(1) for hash indexes). That is why reads become much faster.

Writes, however, must keep the index in sync with the base table. Every INSERT adds a new entry to each relevant index, every UPDATE that touches an indexed column must delete the old key and insert the new one, and every DELETE must remove the corresponding index entries. Each of these operations may trigger page splits, node merges, or bitmap updates, causing extra disk seeks, CPU work, and lock contention. Consequently the overall cost of a write is roughly “base‑table write + index‑maintenance work,” making writes slower as the number and complexity of indexes grow.

TTFT: 0.36s   total: 1.06s


Run it and note the gap between the two numbers: that gap is the user experience you buy with streaming. Each `chunk` is a `ChatCompletionChunk`; the text increment lives at `chunk.choices[0].delta.content` (which is `None` on chunks that carry no text, like the first and last).

## The chunk stream anatomy

Groq follows the OpenAI streaming format: a sequence of `ChatCompletionChunk` objects, each carrying a `delta` (the increment since the last chunk):

| Field | Fires | What's in it |
|---|---|---|
| `delta.role` | first chunk | usually `"assistant"`, content empty |
| `delta.content` | many | the text increment (a token or few) |
| `delta.tool_calls` | when calling tools | incremental fragments of the tool call |
| `finish_reason` | last content chunk | `"stop"`, `"length"`, or `"tool_calls"` |
| `x_groq.usage` | final chunk | token accounting (Groq extension) |

Unlike some providers, Groq returns final token usage on the last chunk under `x_groq.usage`: you get incremental display *and* the billing record in one pass, automatically (no extra request flag needed).


In [4]:
# Same request, inspecting the chunk structure. We accumulate text and read final usage.
text_parts = []
final_usage = None
finish_reason = None

stream = client.chat.completions.create(
    model=MODEL,
    max_tokens=300,
    stream=True,
    messages=[{'role': 'user', 'content': 'Two sentences: what is connection pooling?'}],
)
for chunk in stream:
    # The usage chunk has an empty choices list, so guard before indexing.
    if chunk.choices:
        delta = chunk.choices[0].delta
        if delta.content:
            text_parts.append(delta.content)
        if chunk.choices[0].finish_reason:
            finish_reason = chunk.choices[0].finish_reason
    # Groq attaches usage to the final chunk under x_groq (sent automatically).
    if getattr(chunk, 'x_groq', None) and chunk.x_groq.usage:
        final_usage = chunk.x_groq.usage

full_text = ''.join(text_parts)
print(full_text)
print()
print('finish_reason:', finish_reason)
print('final usage:', final_usage)


Connection pooling is a technique that maintains a set of open, reusable connections (e.g., to a database or network service) so that an application can quickly borrow a connection instead of creating a new one each time. By reusing these pre‑established connections and managing their lifecycle, pooling reduces the overhead of connection setup, improves performance, and conserves system resources.

finish_reason: stop
final usage: CompletionUsage(completion_tokens=178, prompt_tokens=79, total_tokens=257, completion_time=0.383819973, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=95), prompt_time=0.004475546, prompt_tokens_details=None, queue_time=0.243426199, total_time=0.388295519)


Run it and note two things. First, the usage chunk arrives *after* the content is done and carries an empty `choices` list, which is why we guard `if chunk.choices:` before indexing. Second, you assembled the full text by concatenating deltas yourself; there's no separate "final message" object in the OpenAI streaming format, so accumulating as you go is the pattern.

## Streaming + tool use

Tool calls stream too, and this is where it gets fiddly. The tool call arrives in *fragments* across many chunks: the `id` and function `name` land on the first fragment, then the `arguments` string dribbles in piece by piece over subsequent chunks. You accumulate per `index` and parse when the stream finishes. This is how UIs show "Searching for: berlin weath..." while the model is still writing the call.


In [6]:
import json

WEATHER_TOOL = {
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': 'Get current weather for a city. Call for any weather question.',
        'parameters': {
            'type': 'object',
            'properties': {'city': {'type': 'string'}},
            'required': ['city'],
        },
    },
}

# Accumulate tool-call fragments by index. Each index is one tool call.
tool_calls = {}  # index -> {'id': ..., 'name': ..., 'args': ''}

stream = client.chat.completions.create(
    model=MODEL,
    max_tokens=300,
    stream=True,
    tools=[WEATHER_TOOL],
    messages=[{'role': 'user', 'content': 'What is the weather in Berlin?'}],
)
for chunk in stream:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if not delta.tool_calls:
        continue
    for tc in delta.tool_calls:
        slot = tool_calls.setdefault(tc.index, {'id': None, 'name': None, 'args': ''})
        if tc.id:
            slot['id'] = tc.id
        if tc.function and tc.function.name:
            slot['name'] = tc.function.name
            print(f'tool call started: {tc.function.name}')
        if tc.function and tc.function.arguments:
            slot['args'] += tc.function.arguments
            print(f'  partial args so far: {slot["args"]!r}')

# Parse only once the stream is complete — fragments are not valid JSON mid-stream.
for idx, slot in tool_calls.items():
    args = json.loads(slot['args'] or '{}')
    print(f'tool call complete: {slot["name"]}({args})  id={slot["id"]}')


tool call started: get_weather
  partial args so far: '{"city":"Berlin"}'
tool call complete: get_weather({'city': 'Berlin'})  id=fc_53a2013f-25f2-4183-b6b9-06db0f628c6b


Run it and note the `partial args` fragments: they are *not* valid JSON until the stream ends, so never `json.loads` mid-stream. In practice you accumulate manually only when the UI needs live argument display; otherwise you just collect the full arguments string, parse once, and continue the tool loop exactly as in the previous notebook (execute, append the `role: "tool"` result, open a new stream).

## What a real backend needs

In production you're rarely printing to a terminal. You're sitting between the model and a browser. Three concerns the notebook can't show but you should design for:

**1. Forwarding as SSE.** Don't buffer the whole response server-side; re-emit deltas as your own server-sent events (SSE). For the fuller picture (SSE vs. WebSockets vs. long-polling, reconnection semantics, and when to reach for each) see [Real-Time Connection Patterns](https://www.calm.rocks/resources/prepare-interview/system-design/concepts/real-time-connection-patterns/). Sketch (FastAPI-flavored pseudocode; don't run this cell shape in Colab):

```python
@app.post('/chat')
async def chat(req: ChatRequest):
    def gen():
        parts, usage = [], None
        stream = client.chat.completions.create(
            model=MODEL, max_tokens=1000, stream=True, messages=req.messages,
        )
        try:
            for chunk in stream:
                if chunk.choices and chunk.choices[0].delta.content:
                    text = chunk.choices[0].delta.content
                    parts.append(text)
                    yield f'data: {json.dumps({"delta": text})}\n\n'
                if getattr(chunk, 'x_groq', None) and chunk.x_groq.usage:
                    usage = chunk.x_groq.usage
        finally:
            # Runs even if the client disconnected mid-stream:
            record_usage(req.user_id, usage)   # billing truth
        yield 'data: [DONE]\n\n'
    return StreamingResponse(gen(), media_type='text/event-stream')
```

**2. Client disconnects.** Users close tabs mid-answer constantly. Your generator gets cancelled, but the model kept generating and you keep paying. Decide deliberately: abort the upstream request on disconnect (saves tokens, loses the answer) or let it finish and persist the result (costs tokens, enables "resume"). Either way, the `finally` block must still capture usage.

**3. Usage for billing.** The usage chunk arrives *last*. If you only forward text deltas and drop the tail chunk, you've thrown away the bill. Always read the `x_groq.usage` chunk server-side and record it before the request handler exits.


## Exercises

1. Extend the TTFT cell to also record *inter-token gaps* (time between consecutive deltas) and print p50/p95. This is the number that makes streamed output feel smooth or janky.
2. Build `stream_to_list(prompt)` that returns `(chunks, final_usage)`: every text delta in order plus the final usage object. Verify the joined chunks equal the full assembled text.
3. Combine this notebook with the previous one: a `run_agent_streaming` loop that streams each turn, prints text deltas live, then reassembles the tool calls from the fragments and continues until `finish_reason == 'stop'`.
4. Simulate a client disconnect: break out of the chunk loop after the 10th text delta. Confirm you lose the usage chunk (it comes last), then restructure so a `finally` block still records what you streamed so far.
